# A1 | 07–08/10 — Fechar códigos, protocolo e coorte

**Pessoa A.** Converter a pergunta de pesquisa em regras de inclusão, exclusão e desfecho.

**Saída até 08/10:** config de decisões v0.1 (`configs/decisoes.yaml`), tabela de códigos, coorte provisória e waterfall.
**Revisão:** B confere categorias e gráficos; C implementa e testa as regras.

| Passo do plano | Onde está neste notebook |
|---|---|
| 1–3. Fichas/dicionários, pesquisar HOSPITAL, DT_INTERNA, EVOLUCAO, TP_IDADE e códigos; registrar tabela | Seção 1 (manual) + seção 2 |
| 4. Ponto de previsão em DT_INTERNA e campos posteriores | Seção 3 |
| 5–6. Coorte estrita, alvo e categorias separadas | Seções 4 e 6 |
| 7–9. N antes/depois, amostras das exclusões, waterfall | Seções 4 e 5 |
| 10. Atualizar tabela de decisões | Seção 7 |

In [1]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa_a"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "minimas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
SUPERVISED_CONFIG = "configs/supervised.yaml"   # features/alvo/coluna_tempo definidos pelo C
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A1")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

Raiz do repositório: c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Pacote (src/protocolo): c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Config de decisões: c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\configs\decisoes.yaml
[base] C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed\dados-v2\srag.parquet -> 4,666,910 linhas x 54 colunas carregadas (de 198 no arquivo) | leitura compacto (texto em category)
Memória do DataFrame: 0.58 GB


## 1. Fichas e dicionários (manual)
Busque exatamente por: *"SIVEP-Gripe dicionário de dados HOSPITAL EVOLUCAO DT_INTERNA"*, *"SIVEP-Gripe ficha de investigação SRAG hospitalização evolução"*, *"STROBE cohort flow diagram observational study"*.
Para cada campo (HOSPITAL, DT_INTERNA, EVOLUCAO, TP_IDADE, sim/não/ignorado) registre: definição, código, ano, regra de salto, tratamento e evidência — na tabela de decisões (seção 7).

- [ ] Fichas/dicionários dos anos usados localizados
- [ ] Códigos de HOSPITAL, EVOLUCAO, TP_IDADE confirmados por ano

## 2. Colunas essenciais e tabela de códigos

In [2]:
v.a1_columns(df, cfg, rep)
v.a1_code_tables(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a1_tabela_codigos.csv").head(40))

[ok] A1 | Colunas essenciais: 11 colunas presentes
[ok] A1 | Códigos inesperados: todos os códigos dentro do esperado


,codigo,campo,esperado,1917,1918,1941,2009,2011,2012,2016,...,2033,2034,2050,2052,2055,2056,2092,2098,2099,sem_data
0,1,HOSPITAL,True,4,1,4,1,1,2,1,...,3,1,1,1,1,1,1,2,1,72078
1,2,HOSPITAL,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,102956
2,9,HOSPITAL,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10066
3,<vazio>,HOSPITAL,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,119207
4,1,EVOLUCAO,True,1,0,1,1,1,2,0,...,0,0,0,1,0,0,1,1,1,141523
5,2,EVOLUCAO,True,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,79814
6,3,EVOLUCAO,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7784
7,9,EVOLUCAO,True,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,1,0,10928
8,<vazio>,EVOLUCAO,False,3,1,2,0,0,0,0,...,3,0,1,0,1,1,0,0,0,64258
9,1,TP_IDADE,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1695


## 3. Ponto de previsão e campos posteriores
O ponto de previsão proposto é a **internação** (`DT_INTERNA`). Tudo o que só existe depois dela é candidato a vazamento.

In [3]:
print("Ponto de previsão:", cfg["cols"]["dt_interna"])
posteriores = [c for c in cfg["forbidden_vars"] if v.canon(c) in df.columns]
print("Campos potencialmente posteriores presentes na base:")
print(posteriores)

Ponto de previsão: DT_INTERNA
Campos potencialmente posteriores presentes na base:
['EVOLUCAO', 'DT_EVOLUCA', 'DT_ENCERRA', 'UTI', 'DT_ENTUTI', 'DT_SAIDUTI', 'SUPORT_VEN', 'ANTIVIRAL', 'DT_ANTIVIR', 'HOSPITAL', 'TP_ANTIVIR', 'TRAT_COV', 'TIPO_TRAT', 'DT_TRT_COV', 'OUT_ANTIV', 'OUT_TRAT', 'DT_DIGITA']


## 4. Coorte estrita — waterfall
Filtros em sequência: hospitalização confirmada → data válida → período → evolução encerrada → alvo binário.

In [4]:
wf = v.a1_waterfall(df, cfg, rep)
display(wf)

[ok] A1 | Waterfall da coorte: 4,666,910 -> 3,739,615 registros (80.1%)


,ordem,filtro,regra,n_excluido,n_restante,pct_restante
0,0,Base completa,—,0,4666910,100.00
1,1,Hospitalização confirmada,HOSPITAL = 1,232253,4434657,95.02
2,2,Data de internação válida,"DT_INTERNA parseável, ≥ 2009-01-01 e ≤ hoje",72133,4362524,93.48
3,3,Período analítico,"ano(DT_INTERNA) em [2019, 2020, 2021, 2022, 20...",210444,4152080,88.97
4,4,Evolução encerrada,"EVOLUCAO em ['1', '2', '3'] (9/ausente fora)",338815,3813265,81.71
5,5,Alvo binário definido,"EVOLUCAO em {cura, óbito}; óbito por outras ca...",73650,3739615,80.13


## 5. Inspeção das exclusões
Confirme que cada filtro se comporta como a regra escrita (as amostras vêm de `a1_amostra_exclusoes_NN.csv`).

In [5]:
for f in sorted(Path(OUT).glob("a1_amostra_exclusoes_*.csv")):
    print("\n==", f.name)
    display(pd.read_csv(f).head(5))


== a1_amostra_exclusoes_01.csv


,HOSPITAL,DT_INTERNA,EVOLUCAO,TP_IDADE,NU_IDADE_N,DT_NASC,DT_SIN_PRI,DT_NOTIFIC,DT_EVOLUCA,DT_ENCERRA,NU_NOTIFIC
0,NaN,NaN,NaN,3,75,1950-11-02,2026-03-22,2026-03-23,NaN,NaN,31775652598724
1,2.0,NaN,1.0,3,4,2022-03-03,2026-03-23,2026-03-29,2026-03-31,2026-06-11,31781205405890
2,NaN,NaN,1.0,2,8,2025-08-09,2026-04-18,2026-04-28,2026-04-19,2026-06-02,31780414044334
3,2.0,NaN,1.0,3,50,1975-04-08,2026-03-07,2026-03-07,NaN,2026-03-20,31776951547485
4,NaN,NaN,1.0,3,83,1942-12-21,2026-08-10,2026-08-13,NaN,2026-08-13,31786648314248



== a1_amostra_exclusoes_02.csv


,HOSPITAL,DT_INTERNA,EVOLUCAO,TP_IDADE,NU_IDADE_N,DT_NASC,DT_SIN_PRI,DT_NOTIFIC,DT_EVOLUCA,DT_ENCERRA,NU_NOTIFIC
0,1,NaN,1.0,2,8,2025-08-10,2026-04-01,2026-04-15,NaN,2026-04-21,31777549715837
1,1,NaN,1.0,3,77,1949-05-26,2026-07-12,2026-07-12,2026-07-15,2026-07-17,31788874096772
2,1,NaN,1.0,3,7,2018-08-31,2026-01-10,2026-01-12,2026-01-28,2026-02-09,31770647469632
3,1,NaN,NaN,3,3,2023-04-20,2026-09-04,2026-09-09,NaN,NaN,31788975169679
4,1,NaN,1.0,2,3,2025-11-10,2026-02-06,2026-02-07,2026-02-20,2026-06-10,31773940312126



== a1_amostra_exclusoes_03.csv


,HOSPITAL,DT_INTERNA,EVOLUCAO,TP_IDADE,NU_IDADE_N,DT_NASC,DT_SIN_PRI,DT_NOTIFIC,DT_EVOLUCA,DT_ENCERRA,NU_NOTIFIC
0,1,2026-06-11,1.0,2,1,2026-05-02,2026-06-02,2026-06-11,2026-06-19,2026-06-19,31781201987679
1,1,2026-06-15,NaN,3,37,1989-04-04,2026-06-15,2026-06-17,NaN,NaN,31782407823707
2,1,2026-03-10,1.0,3,4,2021-10-06,2026-03-09,2026-03-14,NaN,NaN,31773493207022
3,1,2026-03-18,1.0,1,11,2026-03-07,2026-03-18,2026-03-23,2026-04-14,2026-04-15,31774542694487
4,1,2026-08-18,NaN,3,75,1950-09-09,2026-08-08,2026-08-18,NaN,NaN,31787085791388



== a1_amostra_exclusoes_04.csv


,HOSPITAL,DT_INTERNA,EVOLUCAO,TP_IDADE,NU_IDADE_N,DT_NASC,DT_SIN_PRI,DT_NOTIFIC,DT_EVOLUCA,DT_ENCERRA,NU_NOTIFIC
0,1,2020-06-21,NaN,2,7,2026-01-27,2026-08-08,2026-08-09,NaN,NaN,31787231962439
1,1,2025-12-24,9.0,2,4,2025-12-26,2026-04-09,2026-04-16,NaN,2026-05-25,31776358797785
2,1,2025-10-03,9.0,3,4,2021-09-04,2026-03-01,2026-03-04,NaN,2026-06-03,31780500876680
3,1,2025-12-16,9.0,2,6,2025-10-07,2026-04-08,2026-04-13,NaN,2026-06-03,31780511592426
4,1,2025-12-17,NaN,3,48,1977-06-27,2026-01-10,2026-01-12,NaN,2026-04-02,31768219829649



== a1_amostra_exclusoes_05.csv


,HOSPITAL,DT_INTERNA,EVOLUCAO,TP_IDADE,NU_IDADE_N,DT_NASC,DT_SIN_PRI,DT_NOTIFIC,DT_EVOLUCA,DT_ENCERRA,NU_NOTIFIC
0,1,2025-09-24,3,2,8,2025-08-12,2026-04-11,2026-05-04,2026-04-16,2026-05-04,31777916441450
1,1,2025-12-03,3,2,1,2025-12-03,2026-01-09,2026-01-30,2026-03-11,2026-03-12,31769792468795
2,1,2025-12-18,3,3,77,1948-02-08,2026-01-05,2026-01-06,2026-01-06,2026-01-06,31768488161625
3,1,2025-10-31,3,2,4,2025-10-31,2026-02-04,2026-05-01,2026-04-07,2026-05-01,31777634956473
4,1,2025-12-30,3,3,50,1975-11-28,2026-01-09,2026-01-13,2026-01-21,2026-02-10,31770727069269


## 6. Alvo e categorias separadas
`EVOLUCAO=2` → óbito (1); `EVOLUCAO=1` → cura (0). **Óbito por outras causas (3) fica separado e evolução desconhecida (9/ausente) nunca vira cura.**

In [6]:
v.a1_target(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a1_alvo_por_ano.csv").head(40))

[ok] A1 | Alvo: desconhecido ≠ cura: 9/ausente/3 viram NA; só 1->0 e 2->1


,ano,1,2,3,9,<vazio>,total
0,1917,1,0,0,0,3,4
1,1918,0,0,0,0,1,1
2,1941,1,0,0,1,2,4
3,2009,1,0,0,0,0,1
4,2011,1,0,0,0,0,1
5,2012,2,0,0,0,0,2
6,2016,0,1,0,0,0,1
7,2017,2,0,0,0,0,2
8,2018,10,0,0,0,1,11
9,2019,38292,5221,8,1340,2682,47543


## 7. Tabela de decisões
Ajuste o texto, o `status` (`proposta` → `aprovada`) e o impacto após conferir no dicionário. A mesma tabela alimenta B e C.

In [7]:
E = "A1"
v.log_decision(OUT, "D-01", "Ponto de previsão", "DT_INTERNA (admissão); campos posteriores bloqueados", E,
               impacto="Define quais variáveis podem entrar", evidencia="Dicionário SIVEP-Gripe (completar)")
v.log_decision(OUT, "D-02", "Coorte estrita", "HOSPITAL=1, DT_INTERNA válida, período analítico, evolução encerrada", E,
               impacto="Ver waterfall (a1_waterfall.csv)")
v.log_decision(OUT, "D-03", "Alvo principal", "Óbito SRAG (EVOLUCAO=2) vs cura (EVOLUCAO=1)", E,
               impacto="Alinhar target.positivos do supervised.yaml (G1)")
v.log_decision(OUT, "D-04", "Evolução desconhecida e óbito outras causas",
               "EVOLUCAO=9/ausente e 3 saem do alvo binário; nunca viram cura", E,
               impacto="Exigir filtro EVOLUCAO in {1,2} no loader do C")
v.log_decision(OUT, "D-12", "Features derivadas (criadas no dataset analítico)",
               f"idade_anos (TP_IDADE), faixa_etaria, intervalo sintomas->internação (inválido se ausente/negativo/>{cfg['derivadas']['intervalo_max_dias']} dias) "
               "+ flag, sin/cos da semana ISO da internação", E,
               impacto="Só informação da admissão; o C2 as encontra no dataset (supervised.yaml)")
v.log_decision(OUT, "D-05", "Período analítico",
               f"Desenvolvimento {cfg['dev_years']}; holdout {cfg['holdout_years']}", E,
               impacto="Exclui anos fora do período; alinhar G2")
display(pd.read_csv(Path(OUT) / "tabela_decisoes.csv"))

,id,etapa,tema,decisao,responsavel,data,status,impacto,evidencia
0,D-01,A1,Ponto de previsão,DT_INTERNA (admissão); campos posteriores bloq...,Pessoa A,2026-10-17,proposta,Define quais variáveis podem entrar,Dicionário SIVEP-Gripe (completar)
1,D-02,A1,Coorte estrita,"HOSPITAL=1, DT_INTERNA válida, período analíti...",Pessoa A,2026-10-17,proposta,Ver waterfall (a1_waterfall.csv),NaN
2,D-03,A1,Alvo principal,Óbito SRAG (EVOLUCAO=2) vs cura (EVOLUCAO=1),Pessoa A,2026-10-17,proposta,Alinhar target.positivos do supervised.yaml (G1),NaN
3,D-04,A1,Evolução desconhecida e óbito outras causas,EVOLUCAO=9/ausente e 3 saem do alvo binário; n...,Pessoa A,2026-10-17,proposta,"Exigir filtro EVOLUCAO in {1,2} no loader do C",NaN
4,D-12,A1,Features derivadas (criadas no dataset analítico),"idade_anos (TP_IDADE), faixa_etaria, intervalo...",Pessoa A,2026-10-17,proposta,Só informação da admissão; o C2 as encontra no...,NaN
5,D-05,A1,Período analítico,"Desenvolvimento [2019, 2020, 2021, 2022, 2023,...",Pessoa A,2026-10-17,proposta,Exclui anos fora do período; alinhar G2,NaN


## 8. Exportar a população analítica (v0.1, provisória)
O notebook **C2 exige** `data/analytical/dataset_supervisionado.parquet`. Esta célula o gera a partir da coorte, usando as features, o alvo e a coluna de tempo do `supervised.yaml`. Sai ordenado por tempo e **sem** colunas pós-admissão além das features pedidas. Avise o C quando atualizar.

In [8]:
info = v.a1_export_dataset(df, cfg, rep, SUPERVISED_CONFIG, versao="0.1", excluir_proibidas=False)
print("Arquivo:", info["arquivo"], "|", f"{info['n_linhas']:,} linhas")
print("Anos:", info["anos"])
print("Features proibidas no dataset:", info["features_proibidas_no_dataset"] or "nenhuma")
print("Features ausentes na base:", info["features_ausentes_na_base"] or "nenhuma")
print("Linhas descartadas por data de tempo inválida:", info["descartadas_data_tempo_invalida"])

# Confere o ARQUIVO gravado contra o que o C2 consome (colunas, datas, alvo, check_leakage)
v.validar_contrato_c2(info["arquivo"], SUPERVISED_CONFIG, rep)

[ok] A1 | Features derivadas: criadas no dataset: idade_anos, faixa_etaria
[!!] A1 | Dataset analítico: 3,739,615 linhas v0.1; features ausentes na base: DIAS_SINTOMAS_ATE_INTERNACAO, SEXO_CATEGORIA
Arquivo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\analytical\dataset_supervisionado.parquet | 3,739,615 linhas
Anos: {'2019': 43513, '2020': 979812, '2021': 1480215, '2022': 475980, '2023': 240704, '2024': 229478, '2025': 289913}
Features proibidas no dataset: nenhuma
Features ausentes na base: ['DIAS_SINTOMAS_ATE_INTERNACAO', 'SEXO_CATEGORIA']
Linhas descartadas por data de tempo inválida: 0
[XX] A1-C2 | Colunas exigidas pelo C2: ausentes no dataset: dias_sintomas_ate_internacao, sexo_categoria
[ok] A1-C2 | Coluna de tempo: datetime válido, ordenado: 2019-01-01 -> 2025-12-31
[ok] A1-C2 | Alvo: apenas cura/óbito; positivos [2.0] presentes
[ok] A1-C2 | check_leakage (C2): nenhuma feature está em `proibidas`
[ok] A1-C2 | Tamanho: 3,739,615 linhas


False

## 9. Fechamento

In [9]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

Relatório: outputs\tables\pessoa_a\relatorio_A1.md


,etapa,verificação,status,detalhe
0,A1,Colunas essenciais,OK,11 colunas presentes
1,A1,Códigos inesperados,OK,todos os códigos dentro do esperado
2,A1,Waterfall da coorte,OK,"4,666,910 -> 3,739,615 registros (80.1%)"
3,A1,Alvo: desconhecido ≠ cura,OK,9/ausente/3 viram NA; só 1->0 e 2->1
4,A1,Features derivadas,OK,"criadas no dataset: idade_anos, faixa_etaria"
5,A1,Dataset analítico,ATENÇÃO,"3,739,615 linhas v0.1; features ausentes na ba..."
6,A1-C2,Colunas exigidas pelo C2,FALHA,ausentes no dataset: dias_sintomas_ate_interna...
7,A1-C2,Coluna de tempo,OK,"datetime válido, ordenado: 2019-01-01 -> 2025-..."
8,A1-C2,Alvo,OK,apenas cura/óbito; positivos [2.0] presentes
9,A1-C2,check_leakage (C2),OK,nenhuma feature está em `proibidas`


### Checklist de saída (08/10)
- [ ] Config de decisões v0.1 atualizada (`configs/decisoes.yaml`)
- [ ] Tabela de códigos e tabela de decisões revisadas
- [ ] Waterfall com N excluído/restante/% de cada filtro
- [ ] Amostras das exclusões inspecionadas
- [ ] `data/analytical/dataset_supervisionado.parquet` (v0.1) gerado e C avisado
- [ ] B e C avisados das regras (categorias, alvo, ponto de previsão)